## Johnny - Der Modellrechner

Hinter jedem Computer steckt ein Prozessor, der Befehle holt und ausführt. Das passiert so schnell, dass man nichts davon sieht. **Johnny** ist ein Modellrechner, bei dem wir **jeden einzelnen Schritt sehen** können. Wir können ihm sogar selbst die Befehle geben, so als wären wir das Steuerwerk.


**Johnny öffnen:** [Johnny im Browser](https://dev.inf-schule.de/content/12_rechner/4_johnny/johnny3/) (am besten in einem zweiten Fenster, damit wir dieses Notebook daneben offen lassen können)

**Schreibweise:** `[10]` bedeutet „die Zahl in der Speicherzelle mit der Adresse 10“. Bei Johnny heißen die Speicherzellen auch **Register**.

### 1. Die Bauteile

Johnny hat genau die Bauteile, die wir von der Von-Neumann-Architektur kennen:

| Bauteil bei Johnny | Bedeutung |
|:--|:--|
| **RAM** (Zellen 0 bis 999) | Speicherwerk: hier liegen Daten **und** Programm |
| **Akkumulator** (ACC) | Rechenwerk: enthält das aktuelle Rechenergebnis |
| **Befehlsregister** (*Instruction Register*, INS) | enthält den Befehl, der gerade ausgeführt wird |
| **Befehlszähler** (*Program Counter*, PC) | enthält die Adresse des nächsten Befehls |
| **Control Unit** | Steuerwerk: enthält den **Mikrocode** und steuert den Ablauf |
| **Adressbus** (AB) und **Datenbus** (DB) | Bussystem: hierüber werden Adressen bzw. Inhalte transportiert |

Johnny hat kein Ein- und Ausgabewerk. Die Zahlen schreiben wir direkt in den RAM und lesen die Ergebnisse dort ab.



### 2. Der RAM

Johnny kann in jeder Zelle des RAM eine Zahl von **0 bis 19999** speichern. Mit dem **linken Pfeil** schreiben wir Zahlen in bestimmte Speicherzellen.

Johnny zeigt jede Zahl im Format `XX.YYY` an:

- `XX` ist der **Befehlscode**
- `YYY` ist die **Adresse**, auf die sich der Befehl bezieht

Ist `XX` gleich `00`, ist die Zelle eine ganz normale Zahl (**Daten**). Sonst zeigt Johnny die Bedeutung als Befehl an.

| Anzeige | Bedeutung |
|:--|:--|
| `00.128` | die Zahl 128 |
| `01.014` | `TAKE 14`: kopiere `[14]` in den Akkumulator |
| `02.010` | `ADD 10`: erhöhe den Akkumulator um `[10]` |
| `10.000` | `HLT 0`: halte an |

Alle Befehle von Johnny:

| Code | Befehl | Bedeutung |
|:--|:--|:--|
| 01 | `TAKE x` | kopiert den Wert aus `[x]` in den Akkumulator |
| 02 | `ADD x` | erhöht den Akkumulator um `[x]` |
| 03 | `SUB x` | erniedrigt den Akkumulator um `[x]` |
| 04 | `SAVE x` | speichert den Inhalt des Akkumulators in `[x]` |
| 05 | `JMP x` | springt zu Adresse `x`: der nächste Befehl steht in Zelle `x` |
| 06 | `TST x` | falls `[x] = 0`: überspringt den nächsten Befehl |
| 07 | `INC x` | erhöht `[x]` um 1 |
| 08 | `DEC x` | erniedrigt `[x]` um 1 |
| 09 | `NULL x` | setzt `[x]` auf 0 |
| 10 | `HLT x` | hält das Programm an, `x` spielt keine Rolle |

#### Aufgabe 2.1: Zahl oder Befehl?

Schreibe der Reihe nach diese Zahlen in die Zellen 0 bis 3 des RAM. Überlege vorher bei jeder Zahl, was Johnny wohl anzeigen wird.

| Zelle | Zahl |
|:-:|:-:|
| 0 | 128 |
| 1 | 1014 |
| 2 | 5003 |
| 3 | 10000 |

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
[0] = 128      wird angezeigt als  00.128   (Zahl 128)
[1] = 1014     wird angezeigt als  01.014   (TAKE 14)
[2] = 5003     wird angezeigt als  05.003   (JMP 3)
[3] = 10000    wird angezeigt als  10.000   (HLT 0)
```

Die ersten beiden Stellen (`XX`) sind der Befehlscode, die letzten drei (`YYY`) die Adresse. Die Zahl 1014 ist also nichts anderes als „Code 01 und Adresse 014“.

</details>

#### Aufgabe 2.2: Befehle selbst bauen

Welche Zahl musst du in eine Zelle schreiben, damit Johnny dort den folgenden Befehl anzeigt? Rechne zuerst, prüfe dann in Johnny.

1. `ADD 20`
2. `SAVE 7`
3. `DEC 99`
4. `NULL 5`

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
ADD 20   ->  02.020  ->  2020
SAVE 7   ->  04.007  ->  4007
DEC 99   ->  08.099  ->  8099
NULL 5   ->  09.005  ->  9005
```

Der Befehlscode kommt nach vorne, die Adresse steht immer dreistellig hinten.

</details>

#### Zahl oder Befehl?

Wenn wir in Zelle 1 die Zahl `1014` eingeben, zeigt Johnny sie als Befehl `TAKE 14` an. Was ist sie nun: eine Zahl oder ein Befehl? **Beides!** Ob eine Zelle als Zahl oder als Befehl benutzt wird, entscheidet sich erst, wenn das Programm läuft: Was der Befehlszähler holt, wird als Befehl ausgeführt. Das ist das wichtigste Von-Neumann-Prinzip: **Programm und Daten liegen im selben Speicher.**

### 3. Die Busse - Werte von Hand transportieren

Wir übernehmen Aufgaben des Steuerwerks. **Dazu decken wir die Control Unit ab**. Auch ohne Programm können wir auf die Schaltflächen an RAM, Akkumulator und Bussen klicken. Jede Schaltfläche ist ein **Mikrobefehl**. 

Die ersten Mikrobefehle betreffen den RAM:

```text
db->ram    Inhalt des Datenbus im RAM speichern (an der Adresse auf dem Adressbus)
ram->db    Inhalt des RAM (an der Adresse auf dem Adressbus) auf den Datenbus legen
```

Um den RAM zu benutzen, brauchen wir also **zwei Dinge**: eine Adresse auf dem Adressbus und einen Inhalt auf dem Datenbus. Für den Anfang gibt es bei Johnny zwei Hilfsknöpfe, die im echten Mikrocode nicht verwendet werden können:

```text
ManuellAb   eine Zahl von Hand auf den Adressbus schreiben
ManuellDb   eine Zahl von Hand auf den Datenbus schreiben
```

Adressbus und Datenbus verhalten sich bei Johnny wie kleine Zwischenspeicher: Eine Zahl bleibt auf dem Bus, bis eine andere daraufgelegt wird. (Im echten Rechner sind Busse nur Leitungen und speichern nichts.)

#### Aufgabe 3.1: Zwei Zahlen speichern

Schreibe nur mit `ManuellAb`, `ManuellDb`, `db->ram` und `ram->db` (also ohne den linken Pfeil):

```text
[5] = 42
[6] = 23
```

Notiere dir die Folge deiner Klicks.

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
ManuellDb 42     Zahl 42 auf den Datenbus
ManuellAb 5      Adresse 5 auf den Adressbus
db->ram          [5] = 42

ManuellDb 23
ManuellAb 6
db->ram          [6] = 23
```

Die Reihenfolge von `ManuellDb` und `ManuellAb` ist egal. Es müssen nur beide Busse gefüllt sein, bevor du `db->ram` klickst.

</details>

#### Aufgabe 3.2: Lesen

Lege die Zahl aus Zelle 5 auf den Datenbus. Was steht danach auf dem Datenbus? Und was steht noch in Zelle 5?

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
ManuellAb 5
ram->db          Datenbus enthält jetzt 42
```

Auf dem Datenbus steht 42, und in Zelle 5 steht weiterhin 42. Lesen ändert den RAM nicht, es wird nur eine Kopie auf den Datenbus gelegt.

</details>

#### Aufgabe 3.3: Umkopieren

Kopiere den Inhalt von Zelle 5 in Zelle 9. Verwende dazu nur `ManuellAb`, `ram->db` und `db->ram`. Du brauchst `ManuellDb` nicht.

*Tipp:* Weil Johnny den Wert auf dem Datenbus festhält, kannst du die Adresse zwischendurch wechseln.

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
ManuellAb 5
ram->db          Datenbus enthält 42
ManuellAb 9      der Adressbus zeigt auf Zelle 9, der Datenbus enthält immer noch 42
db->ram          [9] = 42
```

</details>

### 4. Der Akkumulator - Rechnen ohne Programm

Rechnen kann nur das Rechenwerk, bei Johnny also der **Akkumulator**. Die Mikrobefehle dazu:

```text
db->acc    Inhalt des Datenbus in den Akkumulator übernehmen
acc->db    Inhalt des Akkumulators auf den Datenbus legen
plus       Akkumulator um den Inhalt des Datenbus erhöhen
minus      Akkumulator um den Inhalt des Datenbus erniedrigen
acc++      Akkumulator um 1 erhöhen
acc--      Akkumulator um 1 erniedrigen
acc=0      Akkumulator auf 0 setzen
```

Wichtig: `plus` und `minus` rechnen immer mit dem, was gerade **auf dem Datenbus** steht. Gerechnet wird nur im Akkumulator. Alle Werte müssen deshalb erst über den Datenbus zum Akkumulator und das Ergebnis wieder zurück.

#### Aufgabe 4.1: Die erste Addition

Klicke auf `New Ram` und `Reset` und erzeuge dann nochmal den Anfangszustand
 `[5] = 42` und `[6] = 23`. Erzeuge dann nur durch Klicken auf die Schaltflächen diesen Ablauf:

```text
[8] = [5] + [6]
```

Welche Zahl steht danach im Akkumulator, welche in Zelle 8?

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
ManuellAb 5
ram->db          [5] auf den Datenbus
db->acc          Akkumulator = 42
ManuellAb 6
ram->db          [6] auf den Datenbus
plus             Akkumulator = 42 + 23 = 65
acc->db          Ergebnis auf den Datenbus
ManuellAb 8
db->ram          [8] = 65
```

Im Akkumulator und in Zelle 8 steht danach 65.

</details>

#### Aufgabe 4.2: Subtrahieren

Berechne mit den Zahlen aus Zelle 5 und 6 nur durch Klicken:

```text
[9] = [5] - [6]
```

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
ManuellAb 5
ram->db
db->acc          Akkumulator = 42
ManuellAb 6
ram->db
minus            Akkumulator = 42 - 23 = 19
acc->db
ManuellAb 9
db->ram          [9] = 19
```

</details>

#### Aufgabe 4.3: Die Knobelaufgabe

Berechne nur durch Klicken:

```text
[10] = [5] + [5] + [6]
```

Versuche, mit **so wenigen Klicks wie möglich** auszukommen.  

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
ManuellAb 5
ram->db          Datenbus = 42
db->acc          Akkumulator = 42
plus             Akkumulator = 42 + 42 = 84    (der Datenbus enthält immer noch 42!)
ManuellAb 6
ram->db          Datenbus = 23
plus             Akkumulator = 84 + 23 = 107
acc->db
ManuellAb 10
db->ram          [10] = 107
```

Der Trick: `plus` benutzt, was gerade auf dem Datenbus liegt. Die 42 muss deshalb nur einmal geholt werden.

</details>

#### Aufgabe 4.4: Was passiert bei 23 - 42?

Berechne nur durch Klicken (mit `[5] = 42` und `[6] = 23`):

```text
[11] = [6] - [5]
```

Überlege vorher: Das Ergebnis wäre eigentlich -19. Was glaubst du, zeigt Johnny an? Schau dir danach den Akkumulator an.

Probiere noch etwas aus: Schreibe in die Zelle 7 die Zahl 19000 und addiere sie zweimal zum Akkumulator (`plus`, `plus`, mit `[7]` auf dem Datenbus). Was zeigt der Akkumulator?

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
ManuellAb 6
ram->db
db->acc          Akkumulator = 23
ManuellAb 5
ram->db          Datenbus = 42
minus            Akkumulator = 0
acc->db
ManuellAb 11
db->ram          [11] = 0
```

Johnny kennt **keine negativen Zahlen**. Wird mehr abgezogen, als da ist, bleibt der Akkumulator bei **0** stehen. Genauso geht es nach oben nicht über **19999** hinaus: `19000 + 19000` ergibt bei Johnny 19999.

Das gilt auch für `DEC` und `SUB`: Eine Zelle mit 0 bleibt bei `DEC` bei 0.

</details>

### 5. Befehlsregister und Befehlszähler - Ein Befehl von Hand

Bisher haben wir Johnny jeden Mikrobefehl einzeln gegeben. Ein richtiges Programm besteht aber aus Assemblerbefehlen im RAM, 
die Johnny sich selbst holt.  **Decke dafür die Control Unit auf.** 

Dazu brauchen wir zwei weitere Bauteile:

- Der **Befehlszähler** (PC) zeigt auf die Zelle mit dem nächsten Befehl. Er startet bei 0.
- Das **Befehlsregister** (INS) nimmt den geholten Befehl auf.

Die Mikrobefehle dazu:

```text
pc->ab     Befehlszähler auf den Adressbus legen
pc++       Befehlszähler um 1 erhöhen
db->ins    Inhalt des Datenbus ins Befehlsregister übernehmen
ins->ab    Adressteil (YYY) des Befehlsregisters auf den Adressbus legen
ins->mc    zum Mikroprogramm des Befehls springen, dessen Code (XX) im Befehlsregister steht
mc=0       zurück zum Anfang des Mikrocodes
```

Schreibe dieses kleine Programm samt Daten in den RAM (mit dem linken Pfeil):

| Zelle | Zahl | Johnny zeigt |
|:-:|:-:|:--|
| 0 | 1010 | `TAKE 10` |
| 1 | 2011 | `ADD 11` |
| 2 | 4012 | `SAVE 12` |
| 3 | 10000 | `HLT 0` |
| 10 | 5 | Zahl 5 |
| 11 | 7 | Zahl 7 |

#### Aufgabe 5.1: Den Befehl holen (FETCH)

Hole den Befehl aus Zelle 0, indem du diese vier Mikrobefehle der Reihe nach auslöst. Achte nach jedem Schritt darauf, was sich ändert:

```text
pc->ab
ram->db
db->ins
ins->mc
```

Was zeigt das Befehlsregister danach an?

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
pc->ab     Adressbus = 0 (der Befehlszähler steht auf 0)
ram->db    Datenbus = 1010 (der Inhalt von Zelle 0)
db->ins    Befehlsregister = 01.010 = TAKE 10
ins->mc    das Steuerwerk springt zum Mikroprogramm von TAKE
```

Diese vier Mikrobefehle heißen **FETCH** (holen). Sie stehen vor jedem Befehl. Im Befehlsregister steht nun `TAKE 10`.

</details>

#### Aufgabe 5.2: Den Befehl ausführen

Jetzt steht `TAKE 10` im Befehlsregister. Führe ihn aus, indem du die Mikrobefehle seines Mikroprogramms der Reihe nach auslöst. Das Mikroprogramm von `TAKE x` besteht aus diesen Mikrobefehlen:

```text
acc=0
ins->ab
ram->db
plus
pc++
mc=0
```

Überlege vorher: Was steht danach im Akkumulator und im Befehlszähler? Und wozu ist `ins->ab` nötig?

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
acc=0      Akkumulator = 0
ins->ab    Adressbus = 10 (der Adressteil YYY aus 01.010)
ram->db    Datenbus = 5   ([10])
plus       Akkumulator = 0 + 5 = 5
pc++       Befehlszähler = 1, zeigt auf den nächsten Befehl
mc=0       zurück zum Holen des nächsten Befehls
```

Im Akkumulator steht 5 und im Befehlszähler 1. Mit `ins->ab` kommt die Adresse **aus dem Befehl selbst** auf den Adressbus. Dadurch weiß Johnny, welche Zelle `TAKE 10` meint. Der Trick bei `TAKE`: Erst wird der Akkumulator auf 0 gesetzt, dann wird mit `plus` die Zahl vom Datenbus addiert. Das Ergebnis ist die kopierte Zahl.

</details>

#### Aufgabe 5.3: Der zweite Befehl

Führe auf dieselbe Weise den zweiten Befehl (`ADD 11`) aus: erst holen, dann ausführen. Das Mikroprogramm von `ADD x` ist fast das gleiche wie das von `TAKE x`. Überlege zuerst:

- Welcher Mikrobefehl muss weg, damit der Akkumulator nicht vorher gelöscht wird?
- Was steht danach im Akkumulator und im Befehlszähler?

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
pc->ab     Adressbus = 1
ram->db    Datenbus = 2011
db->ins    Befehlsregister = ADD 11
ins->mc
ins->ab    Adressbus = 11
ram->db    Datenbus = 7
plus       Akkumulator = 5 + 7 = 12
pc++       Befehlszähler = 2
mc=0
```

Gegenüber `TAKE` fehlt bei `ADD` nur `acc=0`: Der Akkumulator wird nicht gelöscht, sondern die Zahl vom Datenbus wird zum bisherigen Inhalt addiert. Danach steht 12 im Akkumulator und 2 im Befehlszähler.

</details>

### 6. Die Control Unit - Johnny macht es selbst

Es ist mühsam, jeden Mikrobefehl von Hand zu klicken. Dafür gibt es die **Control Unit**. Sie enthält den **Mikrocode**: für jeden Assemblerbefehl die Liste seiner Mikrobefehle. Am Anfang steht immer der Teil **FETCH**, den wir eben von Hand durchgespielt haben.

Johnny kann nun den Ablauf selbst steuern. In der Oberfläche gibt es Schaltflächen, mit denen Johnny

- **einen einzelnen Mikrobefehl** ausführt,
- **einen ganzen Assemblerbefehl** ausführt,
- das **ganze Programm** selbständig ablaufen lässt.

#### Aufgabe 6.1: Johnny übernimmt

Setze Johnny mit dem **Reset-Button** zurück (Befehlszähler und Akkumulator wieder auf 0; der Inhalt des RAM bleibt erhalten) und lass ihn das Programm aus Abschnitt 5 abarbeiten, aber **einen Mikrobefehl nach dem anderen**. 



Beantworte währenddessen:

1. Welche Mikrobefehle bilden den FETCH-Teil, der vor jedem Assemblerbefehl ausgeführt wird?
2. Welcher der Befehle `TAKE`, `ADD`, `INC` hat das längste Mikroprogramm? Zähle die Mikrobefehle.
3. Was steht am Ende in Zelle 12?

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

1. `pc->ab`, `ram->db`, `db->ins`, `ins->mc`
2. `INC` hat mit 8 Mikrobefehlen das längste Mikroprogramm (`TAKE` hat 6, `ADD` nur 5): `INC x` muss den Wert aus dem RAM in den Akkumulator holen, erhöhen und wieder zurückschreiben.
3. In Zelle 12 steht 12 (5 + 7).

</details>

#### Aufgabe 6.2: Wie funktioniert SAVE?

Schau dir im Mikrocode den Befehl `SAVE x` an, **ohne** ihn ablaufen zu lassen. Kannst du aus den Mikrobefehlen erkennen, was `SAVE` macht? Lass dann das Programm bis zum `SAVE` laufen und prüfe deine Vermutung.

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
ins->ab     die Adresse x kommt auf den Adressbus
acc->db     der Akkumulator kommt auf den Datenbus
db->ram     der Datenbus wird in den RAM geschrieben (an Adresse x)
pc++
mc=0
```

`SAVE x` legt den Akkumulator auf den Datenbus und schreibt ihn in die Zelle `x`.

</details>

### 7. Programme laufen lassen

Wir können Johnny jetzt einzelne Mikrobefehle geben, ganze Befehle ausführen lassen und Programme komplett laufen lassen. 

*Tipp:* Vor jedem neuen Programm können wir mit dem Button **New Ram** (oben ganz links) den RAM leeren, damit keine Reste des vorigen Programms stören. Der **Reset**-Button setzt dagegen nur Befehlszähler und Akkumulator zurück.

#### Aufgabe 7.1: Ein Programm starten

Schreibe dieses Programm in den RAM:

| Zelle | Zahl | Johnny zeigt |
|:-:|:-:|:--|
| 0 | 1004 | `TAKE 4` |
| 1 | 4005 | `SAVE 5` |
| 2 | 10000 | `HLT 0` |
| 4 | 5003 | `JMP 3` |

Was wird Johnny nach dem Start in Zelle 5 anzeigen? Überlege zuerst, dann starte das Programm.

<details>
<summary>Erklärung anzeigen (erst selbst überlegen und ausführen!)</summary>

In Zelle 5 steht danach 5003, und Johnny zeigt dort `JMP 3` an.

`TAKE 4` hat die Zahl 5003 aus Zelle 4 in den Akkumulator kopiert, `SAVE 5` hat sie in Zelle 5 geschrieben. Johnny hat dabei nur eine Zahl kopiert. Dass sie als `JMP 3` angezeigt wird, liegt allein an der Deutung. Die Zahl in Zelle 5 könnte also auch selbst als Befehl ausgeführt werden.

</details>

#### Aufgabe 7.2: Der Fehlerteufel

Diesem Programm ist ein Fehler unterlaufen. Es soll `[12] = [10] + [11]` berechnen:

| Zelle | Zahl | Johnny zeigt |
|:-:|:-:|:--|
| 0 | 1010 | `TAKE 10` |
| 1 | 2010 | `ADD 10` |
| 2 | 4012 | `SAVE 12` |
| 3 | 10000 | `HLT 0` |
| 10 | 5 | Zahl 5 |
| 11 | 7 | Zahl 7 |

Starte das Programm. In Zelle 12 sollte 12 stehen. Was steht dort wirklich? Verbessere dann den Fehler und starte das Programm noch einmal.

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

In Zelle 12 steht 10 statt 12. Nach `TAKE 10` ist der Akkumulator 5, aber `ADD 10` addiert noch einmal die 5 aus Zelle 10 statt der 7 aus Zelle 11. Der Fehler steckt in Zelle 1: dort muss `ADD 11` (also die Zahl 2011) stehen.

Merke: Beim Einzelschritt siehst du den Fehler sofort, wenn du den Akkumulator im Blick behältst.

</details>

#### Aufgabe 7.3: Schleifen und Sprünge

Dieses Programm benutzt die Sprungbefehle `JMP` und `TST`. Schreibe es in den RAM:

| Zelle | Zahl | Johnny zeigt |
|:-:|:-:|:--|
| 0 | 6010 | `TST 10` |
| 1 | 5003 | `JMP 3` |
| 2 | 10000 | `HLT 0` |
| 3 | 7011 | `INC 11` |
| 4 | 8010 | `DEC 10` |
| 5 | 5000 | `JMP 0` |
| 10 | 5 | Zahl 5 |

`TST x` überspringt den nächsten Befehl, falls `[x] = 0`. Sonst geht es ganz normal weiter.

1. Überlege zuerst: Was steht nach dem Start des Programms in den Zellen 10 und 11?
2. Führe das Programm Befehl für Befehl aus, bis `INC 11` zum ersten Mal ausgeführt ist. In welcher Reihenfolge wurden die Zellen abgearbeitet?
3. Starte das Programm. Wie oft wird `INC 11` insgesamt ausgeführt?
4. Ändere `[10]` auf 8 und starte erneut. Was steht jetzt in Zelle 11?

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

1. Das Programm zählt `[10]` herunter und `[11]` hinauf, bis `[10]` gleich 0 ist. Danach steht in Zelle 10 die 0 und in Zelle 11 die 5.
2. Die Zellen werden in der Reihenfolge 0 (`TST 10`), 1 (`JMP 3`), 3 (`INC 11`) abgearbeitet. Weil `[10]` nicht 0 ist, wird `JMP 3` nicht übersprungen.
3. `INC 11` wird 5-mal ausgeführt, einmal pro Durchlauf. Beim 6. Mal ist `[10] = 0`: `TST 10` überspringt dann `JMP 3`, das Programm landet bei `HLT 0` und hält an.
4. Mit `[10] = 8` steht am Ende in Zelle 11 die 8.

</details>

### 8. Eigene Befehle

Wir können den Mikrocode in der Control Unit **erweitern**. Johnny kennt die Befehlscodes 01 bis 10. Einen neuen Befehl legen wir an, indem wir im Mikrocode ein neues Mikroprogramm für ihn aufschreiben.

Ein Mikroprogramm besteht fast immer aus denselben Teilen:

1. Die Mikrobefehle, die den Befehl ausführen.
2. Am Ende `pc++` (nächster Befehl) und `mc=0` (zurück zum FETCH).

Zum Vergleich noch einmal `NULL x` (setzt `[x]` auf 0), wie es im Mikrocode steht:

```text
ins->ab      Adresse x auf den Adressbus
acc=0        Akkumulator auf 0
acc->db      die 0 auf den Datenbus
db->ram      [x] = 0
pc++
mc=0
```

#### Aufgabe 8.1: Der Befehl EINS

Erstelle einen neuen Assemblerbefehl `EINS x`, der die Zelle `[x]` auf 1 setzt:

```text
EINS 10      bedeutet     [10] = 1
```

*Tipp:* Fast alles kannst du von `NULL x` übernehmen. Der Akkumulator muss ja nur noch von 0 auf 1 kommen.

Teste deinen Befehl, indem du ihn in ein kleines Programm einbaust.

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```text
ins->ab
acc=0
acc++
acc->db
db->ram
pc++
mc=0
```

Gegenüber `NULL x` kommt einfach `acc++` dazu. Zum Testen: Hat dein neuer Befehl den Code 11 und steht die Zahl 11010 (`EINS 10`) in Zelle 0 und `HLT` in Zelle 1, dann steht nach dem Start in Zelle 10 die 1.

</details>

### 9. Wichtiges wiederholt:

- Alle Assemblerbefehle sind aus Mikrobefehlen zusammengesetzt.
- Vor jedem Befehl steht der **FETCH**: `pc->ab`, `ram->db`, `db->ins`, `ins->mc`.
- Programm und Daten liegen im selben Speicher: ob eine Zelle Zahl oder Befehl ist, entscheidet nur, wie sie benutzt wird.
- Gerechnet wird nur im Akkumulator.